In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

### 1. Load Dataset

In [ ]:
df = pd.read_csv("insurance.csv")

In [ ]:
print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520
(1338, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB
None
age         0
sex         0
bmi         0
children    0
smoker      0
region     

### 2. Data Preprocessing


In [ ]:
dproc = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Removed {dproc - df.shape[0]} duplicate rows")

#Outlier detection
def remove_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return data[(data[column] >= lower) & (data[column] <= upper)]

dproc = df.shape[0]
df = remove_outliers_iqr(df, "bmi")
print(f"Removed {dproc-df.shape[0]} BMI outlier rows")

Removed 0 duplicate rows
Removed 0 BMI outlier rows


In [ ]:
# Feature engineering
def bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obese"

df["bmi_category"] = df["bmi"].apply(bmi_category)

#Encoding
X = df.drop("charges", axis=1)
y = df["charges"]

numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region", "bmi_category"]
num_transformer = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps= [
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers = [
    ("num", num_transformer, numeric_features),
    ("cat", cat_transformer, categorical_features)
])

### 3. Pipeline Creation

In [ ]:
rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

### 4. Primary Model Selection

### Justification: RandomForestRegressor chosen because:
- Handles non-linear relationships well (charges jump sharply for smokers)
- Robust to outliers and mixed numeric/categorical (after encoding) data
- No need for heavy feature scaling assumptions
- Gives feature_importances_ for interpretability

### 5. Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'bmi', 'children']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['sex', 'smoker', 'region',
                                                   'bmi_category'])])),
                ('model', RandomForestRegressor(n_jobs=-1, random_state=42))])

### 6. Cross Validation

In [ ]:
cv_scores = cross_val_score(
    rf_pipeline, X_train, y_train, cv=5, scoring="r2", n_jobs=-1
)
print(f"R2 scores per fold: {cv_scores}")
print(f"Average R2: {cv_scores.mean():.4f}")
print(f"Std Dev: {cv_scores.std():.4f}")

R2 scores per fold: [0.86218726 0.85054901 0.88412068 0.81414059 0.78221074]
Average R2: 0.8386
Std Dev: 0.0362


### 7. Hyperparameter Tuning

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, 15, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}
grid_search = GridSearchCV (
    estimator = rf_pipeline,
    param_grid = param_grid,
    cv=5, scoring="r2",
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f"Parameters tested: {param_grid}")
print(f"Best Parameters found: {grid_search.best_params_}")
print(f"Best CV R2 Score: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Parameters tested: {'model__n_estimators': [100, 200, 300], 'model__max_depth': [5, 10, 15, None], 'model__min_samples_split': [2, 5, 10], 'model__min_samples_leaf': [1, 2, 4]}
Best Parameters found: {'model__max_depth': 5, 'model__min_samples_leaf': 4, 'model__min_samples_split': 10, 'model__n_estimators': 300}
Best CV R2 Score: 0.8580


### 8. Best Model Selection

In [ ]:
best_pipeline = grid_search.best_estimator_

### 9. Model Performance Evaluation

In [ ]:
y_pred = best_pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

RMSE: 4560.7111
MAE: 2562.0941
R2 Score: 0.8504


### Save Model

In [ ]:
with open("insurance_rf_pipeline.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)

print("✅ Best Random Forest pipeline saved as insurance_rf_pipeline.pkl")

✅ Best Random Forest pipeline saved as insurance_rf_pipeline.pkl
